# ECE326 Assignment 2 - Solutions and Review

These worked solutions accompany the three assignment questions. Run the notebook from top to bottom. NumPy is the only additional package required. Tests deliberately use ordinary loops when comparing against an independent reference; those loops are not part of the constrained array implementations.


## Setup


In [ ]:
import numpy as np


## Q1 - Window totals from one scan
A prefix array with a leading zero makes every window use the same difference, including the first. prefix[j] contains the sum of the first j input values; hence the sum starting at i is prefix[i+k] minus prefix[i].


In [ ]:
def window_totals(x, k):
    prefix = np.zeros(len(x) + 1, dtype=np.int64)
    prefix[1:] = np.add.accumulate(x)
    return prefix[k:] - prefix[:-k]

def best_window(x, k):
    totals = window_totals(x, k)
    start = int(np.argmax(totals))
    return start, int(totals[start])

x = np.array([2, 1, 5, 1, 3], dtype=np.int64)
print("Window totals:", window_totals(x, 3))
print("Best window:", best_window(x, 3))


### Q1(c) - Required result table
| ID | window_totals | best_window |
|----|---------------|-------------|
| W1 | [8, 7, 9] | (2, 9) |
| W2 | [7] | (0, 7) |
| W3 | [2, -1, 3] | (2, 3) |
| W4 | [4] | (0, 4) |
| W5 | [2, 2, 2] | (0, 2) |
| W6 | [-5, -4] | (1, -4) |


### Q1(d) - Short answers
- D1: P[i+k] - P[i].
- D2: P[0]=0, so the first window is P[k]-0, the sum of the first k entries.
- D3: O(n) work and O(n) extra storage, versus O(n*k) work for separately summing the windows.

Subtraction allocates an independent result. The stated bounds keep all prefix sums within int64 range. argmax chooses the first maximum, so ties require no special loop; all-negative inputs also work.


### Q1 verification


In [ ]:
def window_reference(x, k):
    return np.array([sum(int(v) for v in x[i:i+k])
                     for i in range(len(x)-k+1)], dtype=np.int64)

assert best_window(x, 3) == (2, 9)
assert best_window(np.array([2, 0, 2, 0], dtype=np.int64), 2) == (0, 2)
assert best_window(np.array([-4, -1, -3], dtype=np.int64), 2) == (1, -4)
rng = np.random.default_rng(326)
checked = 0
for n in range(1, 26):
    values = rng.integers(-1000, 1001, size=n, dtype=np.int64)
    for k in range(1, n+1):
        before = values.copy()
        expected = window_reference(values, k)
        actual = window_totals(values, k)
        np.testing.assert_array_equal(actual, expected)
        assert actual.shape == (n-k+1,) and actual.dtype == np.int64
        assert not np.shares_memory(actual, values)
        np.testing.assert_array_equal(values, before)
        start, total = best_window(values, k)
        assert type(start) is int and type(total) is int
        expected_start = list(expected).index(max(expected))
        assert (start, total) == (expected_start, int(expected[expected_start]))
        actual[0] += 1
        np.testing.assert_array_equal(values, before)
        checked += 1
# Valid strided inputs are arrays too.
values = np.arange(20, dtype=np.int64)[::2]
np.testing.assert_array_equal(window_totals(values, 4), window_reference(values, 4))
assert best_window(np.full(10000, 1000, dtype=np.int64), 10000) == (0, 10000000)
print(f"Q1 passed: {checked} size/window combinations, examples and storage checks.")


## Q2 - Order-sensitive operator overloading
A class object and an instance of that class have different types. Operator methods act on instances; isinstance accepts subclass instances, whereas an exact type comparison does not.


### Q2(a) - Type relationships
| ID | Left | Right | Reason |
|----|------|-------|--------|
| T1 | True | False | p has exact type ModelPath, which inherits from ResourcePath. |
| T2 | False | True | The class object ModelPath has type type, not ResourcePath. |
| T3 | True | False | ModelPath inherits from ResourcePath, not the reverse. |


### Q2(b) - Implementation


In [ ]:
class ResourcePath:
    def __init__(self, parts):
        self.parts = parts

    def __truediv__(self, other):
        if isinstance(other, ResourcePath):
            suffix = other.parts
        elif isinstance(other, str):
            suffix = (other,)
        else:
            return NotImplemented
        return ResourcePath(self.parts + suffix)

    def __rtruediv__(self, other):
        if isinstance(other, ResourcePath):
            prefix = other.parts
        elif isinstance(other, str):
            prefix = (other,)
        else:
            return NotImplemented
        return ResourcePath(prefix + self.parts)

class ModelPath(ResourcePath):
    pass

p = ModelPath(("models", "small"))
print("O1", (p / "weights").parts)
print("O2", ("cache" / p).parts)
print("O3", (p / ResourcePath(("v1", "weights"))).parts)
try:
    p / 3
except TypeError:
    print("O4 TypeError")
print("O5", p.__truediv__(3) is NotImplemented)
print("O6", p.parts)


### Q2(c) - Fixed results
| ID | Result |
|----|--------|
| O1 | ("models", "small", "weights") |
| O2 | ("cache", "models", "small") |
| O3 | ("models", "small", "v1", "weights") |
| O4 | TypeError |
| O5 | True |
| O6 | ("models", "small") |


### Q2(d) - One sentence
That delegation appends "cache" to self.parts, producing ("models", "small", "cache"), instead of prepending it to produce ("cache", "models", "small").

NotImplemented is a return value that lets Python try the other operand's implementation; it is not NotImplementedError and must not be raised. The tuple-valued parts do not need a deep copy: tuple concatenation and constructing a new ResourcePath preserve both operands. Returning a plain ResourcePath for subclass inputs follows this question's stated contract, not a universal rule for operator design.


### Q2 verification


In [ ]:
assert (isinstance(p, ResourcePath), type(p) is ResourcePath) == (True, False)
assert (isinstance(ModelPath, ResourcePath), isinstance(ModelPath, type)) == (False, True)
assert (issubclass(ModelPath, ResourcePath), issubclass(ResourcePath, ModelPath)) == (True, False)
paths = [ResourcePath(()), ResourcePath(("a",)), ModelPath(("b", "c"))]
for left in paths:
    for right in paths:
        before = left.parts, right.parts
        result = left / right
        assert type(result) is ResourcePath
        assert result is not left and result is not right
        assert result.parts == before[0] + before[1]
        assert (left.parts, right.parts) == before
        reflected = right.__rtruediv__(left)
        assert type(reflected) is ResourcePath and reflected.parts == result.parts
    for name in ["weights", "cache"]:
        before = left.parts
        after = left / name
        prefixed = name / left
        assert type(after) is ResourcePath and type(prefixed) is ResourcePath
        assert after is not left and prefixed is not left
        assert after.parts == before + (name,)
        assert prefixed.parts == (name,) + before
        assert left.parts == before
    for bad in [3, True, 1.5, None, ("x",), ["x"], {}, ResourcePath]:
        assert left.__truediv__(bad) is NotImplemented
        assert left.__rtruediv__(bad) is NotImplemented
        for operation in [lambda: left / bad, lambda: bad / left]:
            try:
                operation()
            except TypeError:
                pass
            else:
                raise AssertionError("Unsupported operand accepted")
assert (p / "weights").parts == ("models", "small", "weights")
assert ("cache" / p).parts == ("cache", "models", "small")
assert (p / ResourcePath(("v1", "weights"))).parts == ("models", "small", "v1", "weights")
assert p.parts == ("models", "small")
print("Q2 passed: type relationships, operand order, subclasses, empty paths,")
print("new results, unchanged operands, and unsupported operand fallback.")


## Q3 - Registration is not wrapping


### Reference implementation


In [ ]:
class Router:
    def __init__(self):
        self.routes = {}

    def route(self, method, path):
        def register(handler):
            key = (method, path)
            if key in self.routes:
                raise ValueError("Route already registered")
            self.routes[key] = handler
            return handler
        return register

    def get(self, path):
        return self.route("GET", path)

    def post(self, path):
        return self.route("POST", path)

    def dispatch(self, method, path, request):
        handler = self.routes[(method, path)]
        return handler(request)


### Why this works
Registration occurs when the decorator is applied during execution of the function definition. The factory captures the router, method and path; its returned decorator receives the function object. No request exists yet, and calling the handler could trigger premature side effects (or fail for lack of request data). Dispatch later performs lookup and invocation with a real request object.

Registration changes the router's mapping, not the function object. Existing aliases still call the handler directly and bypass dispatch. Returning the original handler keeps the decorated name callable and preserves identity. A registration decorator need not create a wrapper.

A path-only key cannot distinguish the read operation from the administrative write operation. A shared registry causes independently constructed applications to interfere: handlers can collide or be exposed in the wrong application's routes. Instance state separates registrations while still allowing each application's handlers to share intentional application state such as the catalogue.


### Service example


In [ ]:
app = Router()
catalogue = ["small"]

@app.get("/v1/models")
def list_models(request):
    return {"models": list(catalogue)}

@app.post("/v1/models")
def add_model(request):
    catalogue.append(request["name"])
    return {"added": request["name"]}

assert catalogue == ["small"]
print(app.dispatch("GET", "/v1/models", {}))
print(app.dispatch("POST", "/v1/models", {"name": "large"}))
print(app.dispatch("GET", "/v1/models", {}))


### Q3(b) - Required trace


In [ ]:
r = Router()
calls = []
def echo(request):
    calls.append(request)
    return request
alias = echo
registered = r.get("/echo")(echo)
print("R1", registered is alias, len(calls))
data = {"x": 1}
print("R2", r.dispatch("GET", "/echo", data) is data, len(calls))
print("R3", alias(data) is data, len(calls))
try:
    r.get("/echo")(lambda request: "replacement")
except ValueError:
    print("R4", "ValueError")
print("R5", r.dispatch("GET", "/echo", data) is data, len(calls))
try:
    r.dispatch("POST", "/echo", data)
except KeyError:
    print("R6", "KeyError", len(calls))


Expected lines: R1 True 0; R2 True 1; R3 True 2; R4 ValueError; R5 True 3; R6 KeyError 3.


### Q3(c) - Short answers
- E1: Registration stores and returns the handler without invoking it, because no request is being processed yet.
- E2: No; alias still refers directly to the original function, so calling it bypasses the router.
- E3: GET and POST handlers for the same path would collide instead of being separate routes.
- E4: Registering a route in one router would affect another router, causing interference or duplicate conflicts.


### Q3 verification
The checks exercise only the documented interface, not the registry's internal name. Distinguish a missing route from an error inside a real handler: both may raise KeyError, but the latter must not be caught and replaced by the router.


In [ ]:
router = Router()
calls = []
request, result = {}, object()
def handler(data):
    calls.append(data)
    return result
alias = handler
register = router.get("/x")
assert calls == []
assert register(handler) is handler and calls == []
assert router.dispatch("GET", "/x", request) is result
assert len(calls) == 1 and calls[0] is request and request == {}
assert alias(request) is result  # Direct call; no route lookup is involved.
for duplicate in [handler, lambda data: None]:
    try:
        register(duplicate)
    except ValueError:
        pass
    else:
        raise AssertionError("Duplicate accepted")
    assert router.dispatch("GET", "/x", request) is result
router.get("/alias")(handler)
router.post("/x")(lambda data: "posted")
assert router.dispatch("GET", "/alias", request) is result
assert router.dispatch("POST", "/x", {}) == "posted"
second = Router()
second.get("/x")(lambda data: "second app")
assert second.dispatch("GET", "/x", {}) == "second app"
assert router.dispatch("GET", "/x", {}) is result
before = len(calls)
for method, path in [("GET", "/X"), ("GET", "/x/"), ("POST", "/missing")]:
    try:
        router.dispatch(method, path, {})
    except KeyError:
        pass
    else:
        raise AssertionError("Unknown route accepted")
assert len(calls) == before
try:
    second.dispatch("POST", "/x", {})
except KeyError:
    pass
else:
    raise AssertionError("Routes leaked between apps")
error = KeyError("raised inside a handler")
def failing(data):
    raise error
router.get("/fail")(failing)
try:
    router.dispatch("GET", "/fail", {})
except KeyError as actual:
    assert actual is error
else:
    raise AssertionError("Exception swallowed")
print("Q3 passed: registration, dispatch, duplicates, identity and isolation.")
